# Modelowanie nieliniowych krzywych popytu detalicznego za pomocą PROC GAM


## Podsumowanie wykonawcze

Ten notatnik wykorzystuje **PROC GAM** do modelowania tygodniowej sprzedaży jednostkowej artykułów spożywczych jako gładkiej, nieliniowej funkcji ceny półkowej, temperatury w sklepie (wskaźnik sezonowości) i wydatków promocyjnych, wraz z parametrycznym efektem flagi promocji. Uogólnione modele addytywne pozwalają kierownikowi kategorii odtworzyć rzeczywiste, krzywoliniowe kształty cenowej elastyczności popytu i sezonowego popytu, których regresja liniowa by nie wychwyciła, wspierając trafniejsze decyzje cenowe i promocyjne.

## Źródła danych

| Zbiór danych | Wiersze | Ziarnistość | Kluczowe zmienne | Opis |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | tydzień | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Syntetyczny tygodniowy szereg danych sprzedażowych dla jednego flagowego sklepu spożywczego na przestrzeni 100 kolejnych tygodni (niemal dwa cykle sezonowe). Wygenerowany wewnątrz notatnika za pomocą `call streaminit(20250531)` i `rand()`. Tygodniowa sprzedaż jednostkowa podlega procesowi popytu Poissona, którego średnia jest sterowana wykładniczą krzywą reakcji na cenę, kwadratowym efektem temperatury/sezonowości szczytującym w okolicach 72°F oraz wklęsłym (pierwiastkowym) wzrostem od wydatków promocyjnych, plus dyskretną flagą tygodnia promocyjnego.

# Modelowanie nieliniowych krzywych popytu detalicznego za pomocą PROC GAM

Popyt detaliczny rzadko reaguje na cenę, pogodę czy wydatki promocyjne w sposób liniowy. Niewielka obniżka ceny produktu podstawowego może ledwie poruszyć wolumen, podczas gdy przekroczenie psychologicznego progu cenowego może wywołać gwałtowny skok; popyt zależny od pogody osiąga szczyt w komfortowym środkowym zakresie i spada na obu skrajnościach; a wzrost sprzedaży dzięki promocji wykazuje malejące przyrosty w miarę zwiększania wydatków.

**PROC GAM** (uogólnione modele addytywne) dopasowuje każdy czynnik za pomocą własnego gładkiego członu splajnowego, dzięki czemu to dane — a nie sztywne założenie liniowości — decydują o kształcie każdej krzywej popytu. Modelujemy tu tygodniową sprzedaż jednostkową dla jednego flagowego sklepu spożywczego na przestrzeni 100 kolejnych tygodni, łącząc parametryczną flagę promocji ze splajnami wygładzającymi dla ceny, temperatury i wydatków promocyjnych, przy odpowiedzi Poissona.

## Krok 1 — Wygenerowanie syntetycznego tygodniowego szeregu sprzedaży

Symulujemy 100 kolejnych tygodni (niemal dwa cykle sezonowe) danych sprzedażowych dla jednego flagowego sklepu. Proces generujący dane jest celowo nieliniowy, abyśmy mogli potwierdzić, że GAM odtwarza realistyczne kształty:

- **Price** napędza wolumen poprzez wykładniczą krzywą reakcji (`exp(-1.1 * Price)`), więc popyt rośnie gwałtownie wraz ze spadkiem ceny.
- **Temperature** pełni rolę wskaźnika sezonowości z kwadratowym szczytem w okolicach 72°F, modelując ruch klientów zależny od komfortu.
- **PromoSpend** daje wklęsły, pierwiastkowy wzrost (malejące przyrosty).
- Dyskretna flaga **Promotion** dodaje parametryczny efekt skokowy w tygodniach objętych promocją.

Tygodniowe `Units` są losowane z rozkładu Poissona, zgodnie z zliczeniowym charakterem sprzedaży jednostkowej.

In [1]:
DANE store_sales;
   CALL streaminit(20250531);
   DŁUGOŚĆ Promotion $3;
   POWTÓRZ Week = 1 TO 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      JEŚLI rand("uniform") < 0.28 WTEDY POWTÓRZ;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      KONIEC;
      PRZECIWNIE POWTÓRZ;
         Promotion  = "No";
         PromoSpend = 0;
      KONIEC;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      WYJŚCIE;
   KONIEC;
WYKONAJ;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Krok 2 — Profilowanie danych symulowanych

Szybkie `PROC MEANS` potwierdza, że zmienne mieszczą się w sensownych zakresach detalicznych przed modelowaniem: liczby jednostek są nieujemnymi liczbami całkowitymi, cena skupia się wokół kilku dolarów, temperatura obejmuje pełny cykl sezonowy, a wydatki promocyjne wynoszą zero w tygodniach bez promocji.

In [2]:
PROCEDURA ŚREDNIE DANE=store_sales n mean MIN MAX maxdec=2;
   ZMIENNA Units Price Temperature PromoSpend;
   ETYKIETA Units="Sprzedaż (szt.)" Price="Cena ($)" Temperature="Temperatura (°F)"
         PromoSpend="Wydatki promo. ($)";
WYKONAJ;

                                                  The MEANS Procedure

 Variable     Label                      N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------------
 Units        Sprzedaż (szt.)          100           7.03        0.00      103.00
 Price        Cena ($)                 100           3.15        2.81        3.48
 Temperature  Temperatura (°F)         100          55.50       22.72       83.49
 PromoSpend   Wydatki promo. ($)       100         128.76        0.00      779.00
 --------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 3 — Dopasowanie pełnego addytywnego modelu popytu

Główny model łączy:

- `param(Promotion)` — parametryczny (liniowy) efekt dla wskaźnika tygodnia promocyjnego, zadeklarowany w instrukcji `CLASS`.
- `spline(Price, df=4)` — sześcienny splajn wygładzający ujmujący krzywą reakcję cenową.
- `spline(Temperature, df=4)` — gładką krzywą sezonowości.
- `spline(PromoSpend, df=3)` — malejące przyrosty wzrostu sprzedaży od promocji.

Ponieważ sprzedaż jednostkowa to zliczenia, określamy `dist=poisson` (funkcja wiążąca log). `method=gcv` pozwala uogólnionej walidacji krzyżowej sterować gładkością każdej składowej bez jawnego narzucania stopni swobody. Instrukcja `OUTPUT` zapisuje prognozy i reszty dla każdej obserwacji do zbioru `gam_fit`.

Procedura raportuje **dewiancję 43,59** wobec **dewiancji zerowej 2041,12** — cztery gładkie i parametryczne czynniki wyjaśniają niemal całą zmienność, którą pozostawiłby model tylko ze stałą — oraz **AIC równe 254,61**. Parametryczne oszacowanie `PROMOTIONYES` jest dodatnie (+0,41 w skali logarytmicznej), potwierdzając wzrost wolumenu dzięki promocji, a splajn ceny niesie silnie ujemny trend liniowy (−1,71), sygnaturę malejącego wraz z ceną popytu.

In [3]:
PROCEDURA gam DANE=store_sales PLOTS=components(CLM commonaxes);
   KLASA Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   ETYKIETA Units="Sprzedaż (szt.)" Promotion="Promocja" Price="Cena ($)"
         Temperature="Temperatura (°F)" PromoSpend="Wydatki promo. ($)";
   WYJŚCIE out=gam_fit predicted residual;
WYKONAJ;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Sprzedaż (szt.)
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Cena ($))                 4.0000 


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Krok 4 — Model skupiony na cenie i sezonowości

Dla bardziej zwięzłego przeglądu cenowego dopasowujemy model ponownie, uwzględniając tylko dwa najistotniejsze decyzyjnie czynniki gładkie — cenę i temperaturę — dając cenie dodatkową elastyczność (`df=5`), aby rozwiązać ewentualne załamanie w pobliżu psychologicznego progu cenowego. Flaga promocji jest zachowana jako kontrola parametryczna.

Usunięcie wydatków promocyjnych podnosi **dewiancję do 62,80** i **AIC do 269,82**, obie wartości wyższe niż 43,59 i 254,61 modelu pełnego. Parametryczny człon `PROMOTIONYES` pochłania tu też więcej sygnału promocyjnego (+1,73 wobec +0,41), ponieważ splajn wydatków nie jest już obecny, by go przenosić. Splajn ceny zachowuje swój ujemny trend (−1,51), więc główna historia popytu pozostaje stabilna między specyfikacjami.

In [4]:
PROCEDURA gam DANE=store_sales PLOTS=components(CLM);
   KLASA Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   ETYKIETA Units="Sprzedaż (szt.)" Promotion="Promocja" Price="Cena ($)"
         Temperature="Temperatura (°F)";
WYKONAJ;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Sprzedaż (szt.)
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Cena ($))                 5.0000         5.0000
Spline(Temperatura (°F))         4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretacja wyników

Tabela **Regression Model Analysis** raportuje trend liniowy w obrębie każdej składowej oraz parametryczny efekt promocji. Dodatni współczynnik `PROMOTIONYES` (+0,41 w modelu pełnym, +1,73 w modelu okrojonym) potwierdza oczekiwany wzrost wolumenu dzięki promocji, natomiast ujemny trend liniowy na splajnie ceny (−1,71 i −1,51) odzwierciedla klasyczny malejący wraz z ceną popyt. Niewielki dodatni człon liniowy splajnu temperatury (+0,03) jest oczekiwany: jego prawdziwa historia to krzywizna wokół szczytu komfortu przy 72°F, której pojedynczy współczynnik liniowy nie może podsumować.

Tabela **Smoothing Model Analysis** raportuje stopnie swobody zużywane przez każdy splajn. Cena i temperatura zużywają po 4 efektywne DF (5 dla ceny w modelu okrojonym), a wydatki promocyjne 3 — każdy znacznie powyżej pojedynczego DF, jakie wykorzystałaby prosta, co dokładnie wyjaśnia, dlaczego regresja liniowa przeoczyłaby te krzywoliniowe zależności.

Tabela **Fit Statistics** pozwala bezpośrednio porównać obie specyfikacje. Pełny czteroczynnikowy model wykazuje dewiancję 43,59 i AIC 254,61 wobec 62,80 i 269,82 modelu okrojonego; oba kryteria faworyzują model pełny, pokazując, że wydatki promocyjne i flaga promocji dodają moc wyjaśniającą ponad samą cenę i temperaturę. W odniesieniu do dewiancji zerowej 2041,12, oba modele wychwytują przytłaczającą większość zmienności popytu.

Razem wzięte, tabele te dają kierownikowi kategorii skwantyfikowaną, opartą na danych historię popytu: wyraźną reakcję cenową informującą o głębokości obniżek, sezonowe okno temperaturowe oraz efekt malejących przyrostów wydatków promocyjnych — znacznie ostrzejszą wskazówkę niż pojedyncze liniowe oszacowanie elastyczności. (`PROC GAM` akceptuje też `plots=components`, aby narysować krzywe częściowej predykcji dla każdego gładkiego członu; powyższe tabele liczbowe składowych są źródłem, z którego te krzywe są rysowane.)